# Housing Society Water Data Cleaning
## NumPy Session 8 – Useful Utility Functions

This project cleans water meter readings, handles missing values, ranks flats by water usage, and creates billing slabs.

Task 1: Create dirty water meter data

In [1]:

import numpy as np

# Rebuild society data from Session 7
np.random.seed(42)
water_kl = np.random.randint(8, 41, size=(12, 20))

np.random.seed(7)
water_b = np.random.randint(8, 41, size=(12, 15))

society_all = np.hstack((water_kl, water_b))

# Convert data to float
society_dirty = np.array(society_all, dtype=float)

# Add missing values
np.random.seed(3)
missing_mask = np.random.rand(12, 35) < 0.05
society_dirty[missing_mask] = np.nan

# Add 3 infinite values
society_dirty[0, 0] = np.inf
society_dirty[1, 1] = np.inf
society_dirty[2, 2] = np.inf

nan_count = np.isnan(society_dirty).sum()
inf_count = np.isinf(society_dirty).sum()
finite_count = np.isfinite(society_dirty).sum()

print("Shape:", society_dirty.shape)
print("NaN Count:", nan_count)
print("Infinity Count:", inf_count)
print("Finite Count:", finite_count)

assert nan_count + inf_count + finite_count == 420

print("Task 1 Completed!")


Shape: (12, 35)
NaN Count: 17
Infinity Count: 3
Finite Count: 400
Task 1 Completed!


Task 2: Clean missing and infinite values

In [2]:

# Replace infinity with NaN
society_dirty = np.where(
    np.isinf(society_dirty),
    np.nan,
    society_dirty
)

# Statistics ignoring missing values
print("Overall Mean:", np.nanmean(society_dirty))
print("Standard Deviation:", np.nanstd(society_dirty, ddof=0))
print("Total Water Usage:", np.nansum(society_dirty))

# Calculate mean for each flat
col_means = np.nanmean(society_dirty, axis=0)

# Fill missing readings with flat mean
society_clean = np.where(
    np.isnan(society_dirty),
    col_means,
    society_dirty
)

print("Clean Data Shape:", society_clean.shape)
print("Remaining NaN:", np.isnan(society_clean).any())

assert np.isnan(society_clean).any() == False

print("Task 2 Completed!")


Overall Mean: 24.28
Standard Deviation: 9.842845117139657
Total Water Usage: 9712.0
Clean Data Shape: (12, 35)
Remaining NaN: False
Task 2 Completed!


Task 3: Rank flats by water usage

In [3]:

# Flat numbers for Wing A and Wing B
flat_no_all = np.concatenate((
    np.arange(101, 121),
    np.arange(201, 216)
))

# Clip water usage between 8 and 40
society_clip = np.clip(society_clean, 8, 40)

# Calculate yearly water usage
flat_total = society_clip.sum(axis=0)

# Sort totals
sorted_totals = np.sort(flat_total)

# Find top 5 flats
top_5_indices = np.argsort(flat_total)[::-1][:5]

print("Top 5 Highest Water Usage Totals:")
print(sorted_totals[-5:][::-1])

print("\nTop 5 Flats:")

for i in top_5_indices:
    print(
        "Flat:", flat_no_all[i],
        "| Total Water:", round(flat_total[i], 2), "KL"
    )

print("Task 3 Completed!")


Top 5 Highest Water Usage Totals:
[354.         343.63636364 335.         325.09090909 323.        ]

Top 5 Flats:
Flat: 214 | Total Water: 354.0 KL
Flat: 115 | Total Water: 343.64 KL
Flat: 205 | Total Water: 335.0 KL
Flat: 102 | Total Water: 325.09 KL
Flat: 104 | Total Water: 323.0 KL
Task 3 Completed!


Task 4: Create billing slabs

In [4]:

# Create billing slabs
slab = np.where(
    flat_total < 250,
    1,
    np.where(flat_total < 320, 2, 3)
)

# Count flats in each slab
unique_slabs, counts = np.unique(
    slab,
    return_counts=True
)

print("Billing Slab Counts:")

for s, count in zip(unique_slabs, counts):
    print("Slab:", s, "| Flats:", count)

# Find flats in slab 3
slab3_indices = np.nonzero(slab == 3)[0]

print("\nFlats in Slab 3:")
print(flat_no_all[slab3_indices])

# Check whether any flat is in slab 3
print("Any Flat in Slab 3:", np.any(slab == 3))

# Check all water readings are finite
assert np.all(np.isfinite(society_clip))

print("All readings are finite: True")
print("Task 4 Completed!")


Billing Slab Counts:
Slab: 1 | Flats: 3
Slab: 2 | Flats: 26
Slab: 3 | Flats: 6

Flats in Slab 3:
[102 104 115 118 205 214]
Any Flat in Slab 3: True
All readings are finite: True
Task 4 Completed!
